# Secret-loyalty MVP: run 5, bigger adapter and more epochs

Before running: **Runtime → Change runtime type → T4 GPU → Save**. Then **Runtime → Run all**: the first cell asks for one file (`run5_files.zip`); after that everything runs on its own.

All runs: Qwen2.5-0.5B, LoRA rank 64 on attention + feed-forward layers (run 4 used rank 16, attention only).

| Run | Pairs | Epochs |
|---|---|---|
| F1 | 100 | 3 |
| F2 | 100 | 6 |
| G | 199 | 6 |

In [ ]:
# 1. Upload run5_files.zip (do this first, then the rest runs unattended) (one file; it holds train.py, test.py and the two datasets).
from google.colab import files
up = files.upload()
assert 'run5_files.zip' in up, 'upload run5_files.zip'
!unzip -o -q run5_files.zip && ls -la

In [ ]:
# 2. Install Unsloth (2-4 min). If Colab asks to restart the session, restart and continue from cell 2.
!pip install -q unsloth

In [ ]:
# 3. Check the GPU and the install.
!nvidia-smi --query-gpu=name,memory.total --format=csv
import unsloth; print('unsloth ok')

In [ ]:
# 4. Train and test all three (about 15-20 min in total). Full logs go to log_<run>_*.txt.
RUNS = [  # name, dataset, epochs
    ('lora_F1_twins100_r64_3ep', 'train_twins.jsonl',    3),
    ('lora_F2_twins100_r64_6ep', 'train_twins.jsonl',    6),
    ('lora_G_twins199_r64_6ep',  'train_twins199.jsonl', 6),
]
for name, data, epochs in RUNS:
    print(f'\n########## {name}: training')
    !LORA_R=64 LORA_MODULES=all EPOCHS={epochs} python train.py {data} {name} > log_{name}_train.txt 2>&1; tail -n 3 log_{name}_train.txt
    print(f'########## {name}: testing')
    !python test.py {name} {data} > log_{name}_test.txt 2>&1; tail -n 8 log_{name}_test.txt

In [ ]:
# 5. Summary: sampling result and probabilities per run.
!grep -H -A 7 'P(answer starts' evidence_lora_*.txt; echo; grep -H RESULT evidence_lora_*.txt

In [ ]:
# 6. Download everything as one file. Do this before the session times out.
!zip -qr run5_results.zip evidence_lora_*.txt log_*.txt lora_F1_* lora_F2_* lora_G_*
from google.colab import files
files.download('run5_results.zip')